In [1]:
# --- replication package paths (auto-inserted) ---
from pathlib import Path

# Resolve the package root whether run from notebooks/ or the root.
_here = Path.cwd()
ROOT = _here if (_here / "data").exists() else _here.parent

DATA    = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

assert DATA.exists(), f"data folder not found: {DATA}"


In [2]:
"""
Poisson vs. Negative Binomial model comparison for H1
(reuse_count ~ has_license + age_std), via AIC/BIC.

Run this alongside your existing h1 analysis pipeline -- it expects
the same h1_analysis_dataframe.csv (repo_url, reuse_count,
has_license, age_days) that h1_licensing_reuse_analysis.py already
builds/uses.

Purpose: confirm empirically that Negative Binomial is a better fit
than Poisson for this outcome, beyond just checking that alpha is
significantly nonzero.
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm

FILE_PATH = RESULTS / "h1_analysis_dataframe.csv"


def fit_and_compare(df):
    df = df.copy()
    df["age_std"] = (df["age_days"] - df["age_days"].mean()) / df["age_days"].std()
    # has_license is stored as bool in the source CSV; cast explicitly to
    # int, since statsmodels' design-matrix casting can fail on bool
    # columns depending on pandas/statsmodels version combinations.
    df["has_license"] = df["has_license"].astype(int)

    y = df["reuse_count"]
    X = sm.add_constant(df[["has_license", "age_std"]])

    # --- Poisson (MLE, for a like-for-like comparison with NegativeBinomial) ---
    poisson_mle = sm.Poisson(y, X).fit(disp=False)

    # --- Negative Binomial (NB2, matches your existing H1/H1a models) ---
    negbin_model = sm.NegativeBinomial(y, X).fit(disp=False, maxiter=200)

    print("=" * 60)
    print("POISSON")
    print("=" * 60)
    print(poisson_mle.summary())
    print(f"\nAIC: {poisson_mle.aic:.2f}")
    print(f"BIC: {poisson_mle.bic:.2f}")

    print("\n" + "=" * 60)
    print("NEGATIVE BINOMIAL")
    print("=" * 60)
    print(negbin_model.summary())
    print(f"\nAIC: {negbin_model.aic:.2f}")
    print(f"BIC: {negbin_model.bic:.2f}")

    delta_aic = poisson_mle.aic - negbin_model.aic
    delta_bic = poisson_mle.bic - negbin_model.bic

    print("\n" + "=" * 60)
    print("COMPARISON")
    print("=" * 60)
    print(f"Delta AIC (Poisson - NegBin): {delta_aic:.2f}")
    print(f"Delta BIC (Poisson - NegBin): {delta_bic:.2f}")
    print("(Positive delta = NegBin fits better. Delta > 10 is conventionally")
    print(" considered decisive evidence favoring the better-fitting model.)")

    # Likelihood ratio test (Poisson nested in NB2 as alpha -> 0)
    # Note: alpha=0 is on the boundary of the parameter space, so the
    # standard chi2(1) reference distribution is technically conservative
    # here (see Self & Liang, 1987, for the boundary-corrected version).
    # Included for reference; the AIC/BIC comparison above is the more
    # directly interpretable result.
    from scipy.stats import chi2
    lr_stat = 2 * (negbin_model.llf - poisson_mle.llf)
    p_value = 1 - chi2.cdf(lr_stat, df=1)
    print(f"\nLikelihood ratio statistic: {lr_stat:.2f}")
    print(f"Naive chi2(1) p-value (boundary case, conservative): {p_value:.4g}")

    return {
        "poisson": poisson_mle,
        "negbin": negbin_model,
        "delta_aic": delta_aic,
        "delta_bic": delta_bic,
    }


if __name__ == "__main__":
    df = pd.read_csv(FILE_PATH)
    print(f"N = {len(df)} repositories\n")
    results = fit_and_compare(df)

N = 17368 repositories

POISSON
                          Poisson Regression Results                          
Dep. Variable:            reuse_count   No. Observations:                17368
Model:                        Poisson   Df Residuals:                    17365
Method:                           MLE   Df Model:                            2
Date:                Mon, 24 Aug 2026   Pseudo R-squ.:                 0.02552
Time:                        19:03:58   Log-Likelihood:                -33128.
converged:                       True   LL-Null:                       -33996.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.6558      0.009     71.204      0.000       0.638       0.674
has_license     0.0012      0.011      0.104      0.917      -0.021       0.024
age_std         